## Setup

### Imports

In [4]:
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

In [5]:
# load precursor and protein data from diann with alphapepttools functions

In [6]:
# load h5ad data saved by alphapepttools
prec_adata = ad.read_h5ad("../data/albrecht.precursors.h5ad")
prot_adata = ad.read_h5ad("../data/albrecht.proteins.h5ad")

In [7]:
msdata = md.MuData(
    # These are the raw data levels
    {
        "protein_level": prot_adata,
        #"peptide_level": ad.AnnData(...),
        "precursor_level": prec_adata,
    },
    # This stores the feature mapping as adjacency matrix of a DAG
    #varp = csr_matrix(...)
)

/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/Users/mcthielert/miniforge3/envs/msmudata_dev/lib/python3.14/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## implement on functions

In [12]:
def get_unique_mappings(psm_table: pd.DataFrame, feature_level_names: list[str]) -> pd.DataFrame:
    """
    Get unique mappings from PSM table for specified feature levels.
    
    Parameters:
    - psm_table: DataFrame containing PSM data with columns for each feature level.
    - feature_level_names: List of column names corresponding to feature levels (e.g., ["Precursor", "Protein"]).
    
    Returns:
    - DataFrame with unique mappings between the specified feature levels.
    """
    # Select only the relevant columns for mapping
    mapping_df = psm_table[feature_level_names].drop_duplicates()
    
    return mapping_df

In [13]:
test_map = get_unique_mappings(prec_adata.var, ["Protein.Group"])

In [14]:
test_map

,Protein.Group
AAAAAAALQAK2,P36578
AAAATGTIFTFR2,P05154
AAAFEEQENETVVVK2,Q9Y490
AAAFLGDIALDEEDLR3,P13497
AAAGEFADDPC(UniMod:4)SSVK2,P35221;P35221-2
...,...
YPVVPVHLDTTI2,P12724
YQAVTATLEEK2,P40429
YQFFVYLQEGK2,Q96S96
YREWHHFLVVNMK4,P30086


In [ ]:
def sparse_matrix_mapping(mapping_df: pd.DataFrame) -> csr_matrix:
    """
    Create a square sparse adjacency matrix for varp from a feature-level mapping.

    Parameters
    ----------
    mapping_df : pd.DataFrame
        DataFrame where the index contains source features (e.g., precursors)
        and each column contains target features (e.g., protein groups).
        Produced by get_unique_mappings().

    Returns
    -------
    csr_matrix
        Square adjacency matrix of shape (n_total, n_total) where
        n_total = n_source + n_target features.
    """
    source_features = mapping_df.index.unique()

    all_row_idx = []
    all_col_idx = []

    target_offset = len(source_features)
    source_lookup = pd.Index(source_features)

    for col in mapping_df.columns:
        target_features = mapping_df[col].unique()
        target_lookup = pd.Index(target_features)

        # Source index: position in source_features
        row = source_lookup.get_indexer(mapping_df.index)
        # Target index: position in target_features, offset by n_source
        col_vals = target_lookup.get_indexer(mapping_df[col]) + target_offset

        # Filter out any unmatched (-1) entries
        valid = (row >= 0) & (col_vals >= target_offset)
        all_row_idx.append(row[valid])
        all_col_idx.append(col_vals[valid])

        # Update offset for next target level
        target_offset += len(target_features)

    n_total = target_offset
    row_idx = np.concatenate(all_row_idx)
    col_idx = np.concatenate(all_col_idx)
    data = np.ones(len(row_idx), dtype=np.float32)

    mapping_matrix = csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(n_total, n_total),
    )
    return mapping_matrix

In [16]:
matrix_varp = sparse_matrix_mapping(test_map)

Source variables
Index(['AAAAAAALQAK2', 'AAAATGTIFTFR2', 'AAAFEEQENETVVVK2',
       'AAAFLGDIALDEEDLR3', 'AAAGEFADDPC(UniMod:4)SSVK2',
       'AAAIGIDLGTTYSC(UniMod:4)VGVFQHGK3', 'AAAPAPVSEAVC(UniMod:4)R2',
       'AAAPNTPK1', 'AAATLMSER2', 'AAATPESQEPQAK2',
       ...
       'YGPVFSFTMVGK2', 'YLNGLGR2', 'YLYTLEK2', 'YLYTLNDNAR2', 'YNLGLDLR2',
       'YPVVPVHLDTTI2', 'YQAVTATLEEK2', 'YQFFVYLQEGK2', 'YREWHHFLVVNMK4',
       'YVLMVVASDR2'],
      dtype='object', length=2161)
Target variables
['P36578', 'P05154', 'Q9Y490', 'P13497', 'P35221;P35221-2', ..., 'P12724', 'P40429', 'Q96S96', 'P30086', 'Q6V0I7;Q6V0I7-3']
Length: 2161
Categories (2161, object): ['A0A0A0MRZ8;P04433', 'A0A0A0MS15', 'A0A0A0MT31', 'A0A0A0MT36', ..., 'S4R471', 'V9GYG9', 'V9GYJ8', 'V9GYS1']


NameError: name 'global_var_names' is not defined

In [32]:
matrix_varp

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (2161, 2161)>

In [23]:
matrix_varp

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (2161, 2161)>